<a href="https://colab.research.google.com/github/Abdullah-Farooq292/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-Farooq292/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For my lane (Lane 2: Refresh/Content Opportunity Scoring), one row = one content page, for one client, on one day — taken from fact_content_daily_performance. I'll work with a mid-panel month, month=2026-03, to build and test my logic safely before touching the final month.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as cnt
    FROM read_parquet('{base}')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate rows found (should be empty if grain is correct):")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows found (should be empty if grain is correct):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, cnt]
Index: []


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
- Features (known before the decision, safe to use): gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, scroll_events, sessions_organic
- Label/proxy: not directly available in this table — I'll need to derive a trend/decline label myself from gsc_impressions or ga4_sessions changes over time (unlike the starter CSV, this warehouse table doesn't ship a pre-computed trend_direction).
- Context (background, not a feature): client_hash_id, content_hash_id, report_date, month — used for joining/grouping/time windows, not as direct model inputs
- Excluded: any FlyRank product decision flags (health_score, priority_score, action_type) — not present in this table at all, and would be leakage if reconstructed and reused as a feature.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
preview = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions
    FROM read_parquet('{base}')
    LIMIT 5
""").df()
print(preview)

  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   gsc_impressions  gsc_clicks  gsc_avg_position  ga4_sessions  
0               20           0          3.350000          <NA>  
1                1           0          0.000000          <NA>  
2              125           1          4.928000          <NA>  
3                7           0          4.000000          <NA>  
4               11           0          2.272727          <NA>  


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The queries below verify: (1) total row count and date span for month=2026-03, confirming this slice matches the window I claimed, and (2) how many rows have real GA4 data available, since the preview showed some sessions values are missing (NA).
Verification results: month=2026-03 contains 9,841,378 rows spanning exactly 2026-03-01 to 2026-03-31, matching my claimed window. However, only 413,966 rows (about 4%) have real GA4 (analytics/session) data available — the rest only have search console data. This means any feature relying on ga4_sessions or engagement will only be usable for a small fraction of pages, and I need to check per-client GA4 availability before relying on those columns broadly.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Row count + date span check
span_check = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('{base}')
""").df()
print("Row count and date span:")
print(span_check)

# Availability check — how many rows actually have GA4 data
availability_check = con.sql(f"""
    SELECT COUNT(*) as rows_with_ga4
    FROM read_parquet('{base}')
    WHERE ga4_data_available IS TRUE
""").df()
print("Rows with real GA4 data available:")
print(availability_check)

Row count and date span:
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with real GA4 data available:
   rows_with_ga4
0         413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data has a few real limits I need to respect:
1. Unbalanced history: different clients have different amounts of tracking history — some have GA4 data, some don't (only ~4% of rows in this month have real GA4 data, as shown above).
2. GSC-only early rows: many rows only have search console data, meaning engagement-based features (sessions, scroll rate) simply can't be computed for most of the dataset.
3. Window overlap risk: if I ever build a future-looking label, I need to make sure my feature window (e.g., prior 90 days) never overlaps with my target window (e.g., next 30 days) — otherwise the model would "see the future" and produce fake-looking results.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [done ] Every section above is filled — markdown thinking AND the code that backs it
- [done ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [done ] No client names, URLs, or private queries anywhere
- [done ] My claims use careful words: observed, measured, directional, decision-support
- [done ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.